# Notebook 12 – String Operations

Text data almost never arrives clean. Names come in inconsistent casing, emails have stray whitespace, feedback comments are free-form, and useful information (like a phone number or a keyword) is often buried inside a longer string. Pandas **vectorized string methods** — accessed through the **.str** accessor — let us clean and extract information from an entire column at once, without writing a Python loop.

### The Dataset We'll Use Throughout

We continue with the same online retail customer story from Notebooks 7–11, now adding a few deliberately **messy** text columns: customer_name_raw (inconsistent casing/spacing), **email** (contact info), and **feedback_notes** (free-form customer feedback).

- **Business angle:** Customer support wants clean, standardized names and emails, and wants to quickly find all feedback mentioning "delivery" complaints.
- **AI/ML angle:** Before feedback text can be used in a churn model (e.g., as a "complaint flag" feature), it needs to be cleaned and searched for specific keywords/patterns — a common NLP preprocessing step.

Let's rebuild the dataset with these text columns, then work through each string operation.

In [1]:
import pandas as pd
import numpy as np
data = {
    "customer_id":   [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    "customer_name_raw": ["  Aarav Sharma", "PRIYA verma  ", "Rahul Nair", "sneha Gupta  ",
                           "  VIKRAM singh", "Ananya Rao", "karthik IYER  ", "  Divya Menon",
                           "Manoj  Kumar", "LAKSHMI pillai  "],
    "region":        ["South", "North", "South", "West", "East",
                       "South", "North", "West", "East", "South"],
    "membership":    ["Gold", "Silver", "Gold", "Bronze", "Silver",
                       "Gold", "Bronze", "Gold", "Silver", "Bronze"],
    "email": ["aarav.sharma@GMAIL.com", "priya_verma@yahoo.com ", " rahul.nair@gmail.com",
              "sneha.gupta@OUTLOOK.com", "vikram.singh@gmail.com", "ananya.rao@yahoo.com",
              "karthik.iyer@GMAIL.com", "divya.menon@outlook.com", "manoj.kumar@gmail.com ",
              " lakshmi.pillai@YAHOO.com"],
    "feedback_notes": [
        "Great service, fast delivery! Call me at 98765-43210 if needed.",
        "Delivery was delayed by 3 days, not happy.",
        "Product quality is excellent, will order again.",
        "Very disappointed - delivery never arrived. Contact: 91234-56789",
        "Support team was helpful, resolved my issue quickly.",
        "Amazing experience, quick delivery every time!",
        "delivery delayed AGAIN. This is the second time.",
        "No complaints, everything works as expected.",
        "Packaging was damaged but delivery was on time.",
        "Terrible support, delivery delayed, want a refund. Reach me at 90000-11111"
    ]
}
df = pd.DataFrame(data)
df

,customer_id,customer_name_raw,region,membership,email,feedback_notes
0,101,Aarav Sharma,South,Gold,aarav.sharma@GMAIL.com,"Great service, fast delivery! Call me at 98765..."
1,102,PRIYA verma,North,Silver,priya_verma@yahoo.com,"Delivery was delayed by 3 days, not happy."
2,103,Rahul Nair,South,Gold,rahul.nair@gmail.com,"Product quality is excellent, will order again."
3,104,sneha Gupta,West,Bronze,sneha.gupta@OUTLOOK.com,Very disappointed - delivery never arrived. Co...
4,105,VIKRAM singh,East,Silver,vikram.singh@gmail.com,"Support team was helpful, resolved my issue qu..."
5,106,Ananya Rao,South,Gold,ananya.rao@yahoo.com,"Amazing experience, quick delivery every time!"
6,107,karthik IYER,North,Bronze,karthik.iyer@GMAIL.com,delivery delayed AGAIN. This is the second time.
7,108,Divya Menon,West,Gold,divya.menon@outlook.com,"No complaints, everything works as expected."
8,109,Manoj Kumar,East,Silver,manoj.kumar@gmail.com,Packaging was damaged but delivery was on time.
9,110,LAKSHMI pillai,South,Bronze,lakshmi.pillai@YAHOO.com,"Terrible support, delivery delayed, want a ref..."


**Output Explanation:** `customer_name_raw` has inconsistent capitalization and stray leading/trailing spaces, `email` has mixed casing and extra whitespace, and `feedback_notes` is free-form text containing keywords (like "delivery") and embedded phone numbers. This is realistic, messy real-world data — exactly what the rest of this notebook will clean up.

## 1. .str.lower()

### Concept Explanation
**.str.lower()** converts every string in a Series to **lowercase**. This is one of the most common first steps in text cleaning, because it makes comparisons, searches, and grouping **case-insensitive** — "Delivery", "DELIVERY", and "delivery" should all be treated as the same word.

### Business + AI/ML Example
Customer emails were entered with inconsistent casing (GMAIL.com vs gmail.com). Before using email domain to group customers (e.g., "how many Gmail users do we have?"), everything needs to be lowercased so the same domain isn't counted as two different ones.

In [2]:
df["email_clean"] = df["email"].str.lower()
df[["email", "email_clean"]]

,email,email_clean
0,aarav.sharma@GMAIL.com,aarav.sharma@gmail.com
1,priya_verma@yahoo.com,priya_verma@yahoo.com
2,rahul.nair@gmail.com,rahul.nair@gmail.com
3,sneha.gupta@OUTLOOK.com,sneha.gupta@outlook.com
4,vikram.singh@gmail.com,vikram.singh@gmail.com
5,ananya.rao@yahoo.com,ananya.rao@yahoo.com
6,karthik.iyer@GMAIL.com,karthik.iyer@gmail.com
7,divya.menon@outlook.com,divya.menon@outlook.com
8,manoj.kumar@gmail.com,manoj.kumar@gmail.com
9,lakshmi.pillai@YAHOO.com,lakshmi.pillai@yahoo.com


**Output Explanation:** Every email is now fully lowercase — aarav.sharma@GMAIL.com became aarav.sharma@gmail.com. Note the stray whitespace is still there; .str.lower() only changes case, it doesn't trim spaces (that's **.str.strip()**, covered next).

## 2. .str.upper()

### Concept Explanation
.str.upper() is the mirror image of .str.lower() — it converts every string to **uppercase**. It's commonly used for generating standardized codes/IDs, or for display purposes where a business convention requires all-caps text (e.g., region codes on a shipping label).

### Business + AI/ML Example
The logistics team wants a standardized, all-caps **region code** column for shipping labels, so "south" and "South" both print consistently as "SOUTH".

In [16]:
df["region_code"] = df["region"].str.upper()
df[["region", "region_code"]].head(5)

,region,region_code
0,South,SOUTH
1,North,NORTH
2,South,SOUTH
3,West,WEST
4,East,EAST


**Output Explanation:** Every value in region was converted to uppercase in the new region_code column — a simple, direct use of .str.upper() for a formatting/display requirement.

## 3. `.str.strip()`

### Concept Explanation
.str.strip() removes leading and trailing whitespace (spaces, tabs, newlines) from each string — it does **not** touch whitespace in the middle. This is essential before comparing, grouping, or joining on text columns, since " Gold" and "Gold" look identical to a human but are different strings to pandas. Related variants: .str.lstrip() (left only) and .str.rstrip() (right only).

### Business + AI/ML Example
Both customer_name_raw and email have stray leading/trailing spaces from a messy data-entry system. Before using these columns for matching or deduplication, the whitespace must be removed.

In [19]:
df["customer_name_clean"] = df["customer_name_raw"].str.strip()
df["email_clean"] = df["email_clean"].str.strip()

In [20]:
df["customer_name_clean"] = df["customer_name_raw"].str.strip()
df["email_clean"] = df["email_clean"].str.strip()
df[["customer_name_raw", "customer_name_clean", "email_clean"]]   

,customer_name_raw,customer_name_clean,email_clean
0,Aarav Sharma,Aarav Sharma,aarav.sharma@gmail.com
1,PRIYA verma,PRIYA verma,priya_verma@yahoo.com
2,Rahul Nair,Rahul Nair,rahul.nair@gmail.com
3,sneha Gupta,sneha Gupta,sneha.gupta@outlook.com
4,VIKRAM singh,VIKRAM singh,vikram.singh@gmail.com
5,Ananya Rao,Ananya Rao,ananya.rao@yahoo.com
6,karthik IYER,karthik IYER,karthik.iyer@gmail.com
7,Divya Menon,Divya Menon,divya.menon@outlook.com
8,Manoj Kumar,Manoj Kumar,manoj.kumar@gmail.com
9,LAKSHMI pillai,LAKSHMI pillai,lakshmi.pillai@yahoo.com


**Output Explanation:** Comparing customer_name_raw to customer_name_clean, the surrounding spaces are gone (e.g., "  Aarav Sharma" → "Aarav Sharma"), and email_clean no longer has leading/trailing whitespace either. This single step prevents silent bugs later — e.g., a `groupby("email_clean") would previously have treated " x@gmail.com" and "x@gmail.com" as two different customers.

## 4. .str.split()

### Concept Explanation
.str.split() breaks each string into a **list of pieces** based on a delimiter (space, comma, @, etc.). Passing expand=True turns those pieces directly into separate DataFrame columns instead of leaving them as lists — extremely useful for pulling apart a "full name" into first/last name, or an email into username and domain.

### Business + AI/ML Example
CRM wants customer_name_clean split into separate first_name and last_name columns, and marketing wants the **email domain** isolated (to analyze "which email providers do our customers use?").

In [13]:
name_parts = df["customer_name_clean"].str.split(" ", n=1, expand=True)
df["first_name"] = name_parts[0].str.capitalize()
df["last_name"] = name_parts[1].str.capitalize()
df[["customer_name_clean", "first_name", "last_name"]]

,customer_name_clean,first_name,last_name
0,Aarav Sharma,Aarav,Sharma
1,PRIYA verma,Priya,Verma
2,Rahul Nair,Rahul,Nair
3,sneha Gupta,Sneha,Gupta
4,VIKRAM singh,Vikram,Singh
5,Ananya Rao,Ananya,Rao
6,karthik IYER,Karthik,Iyer
7,Divya Menon,Divya,Menon
8,Manoj Kumar,Manoj,kumar
9,LAKSHMI pillai,Lakshmi,Pillai


**Output Explanation:** str.split(" ", n=1, expand=True) split each name on the **first** space only (n=1), producing two columns which we then assigned to first_name and last_name. .str.capitalize() was chained on afterward to fix any remaining casing inconsistencies (e.g., "verma" → "Verma").

In [14]:
df["email_domain"] = df["email_clean"].str.split("@").str[1]
df[["email_clean", "email_domain"]]

,email_clean,email_domain
0,aarav.sharma@gmail.com,gmail.com
1,priya_verma@yahoo.com,yahoo.com
2,rahul.nair@gmail.com,gmail.com
3,sneha.gupta@outlook.com,outlook.com
4,vikram.singh@gmail.com,gmail.com
5,ananya.rao@yahoo.com,yahoo.com
6,karthik.iyer@gmail.com,gmail.com
7,divya.menon@outlook.com,outlook.com
8,manoj.kumar@gmail.com,gmail.com
9,lakshmi.pillai@yahoo.com,yahoo.com


**Output Explanation:** Splitting each email on @ produces a list like [aarav.sharma, gmail.com]; .str[1] then grabs the second element (the domain) from every row's list at once. This gives marketing a clean email_domain column ready for a value_counts() breakdown of email providers.

## 5. .str.contains()

### Concept Explanation
.str.contains() checks whether each string **contains a given substring or pattern**, returning True/False for every row — perfect for filtering. It's case-sensitive by default; pass case=False for a case-insensitive search, and na=False to safely treat missing values as "not matching" instead of raising an error.

### Business + AI/ML Example
Support wants to instantly pull up every customer whose feedback mentions **"delivery"** (regardless of capitalization), since delivery complaints are a known churn driver. This becomes a has_delivery_complaint flag for the churn model.

In [7]:
delivery_mentions = df["feedback_notes"].str.contains("delivery", case=False, na=False)
df[delivery_mentions][["customer_name_clean", "feedback_notes"]]

,customer_name_clean,feedback_notes
0,Aarav Sharma,"Great service, fast delivery! Call me at 98765..."
1,PRIYA verma,"Delivery was delayed by 3 days, not happy."
3,sneha Gupta,Very disappointed - delivery never arrived. Co...
5,Ananya Rao,"Amazing experience, quick delivery every time!"
6,karthik IYER,delivery delayed AGAIN. This is the second time.
8,Manoj Kumar,Packaging was damaged but delivery was on time.
9,LAKSHMI pillai,"Terrible support, delivery delayed, want a ref..."


**Output Explanation:** str.contains("delivery", case=False) matched the word regardless of case — catching both "delivery" and "Delivery" (and "delivery" in "delivery delayed AGAIN"). This filtered view shows every customer whose feedback references delivery, ready for the support team to review or the model to flag.

In [4]:
df["mentions_delivery"] = df["feedback_notes"].str.contains("delivery", case=False, na=False)
df["mentions_delay_or_disappointed"] = df["feedback_notes"].str.contains(
    "delay|disappoint|terrible", case=False, na=False
)
df[["customer_name_clean", "mentions_delivery", "mentions_delay_or_disappointed"]]

,customer_name_clean,mentions_delivery,mentions_delay_or_disappointed
0,Aarav Sharma,True,False
1,PRIYA verma,True,True
2,Rahul Nair,False,False
3,sneha Gupta,True,True
4,VIKRAM singh,False,False
5,Ananya Rao,True,False
6,karthik IYER,True,True
7,Divya Menon,False,False
8,Manoj Kumar,True,False
9,LAKSHMI pillai,True,True


**Output Explanation:** The first flag is a simple keyword check. The second uses the **|** (OR) operator inside the pattern to match **any** of three negative keywords at once — str.contains() accepts basic regex patterns directly, which is a natural bridge into the full Regular Expressions section below.

## 6. .str.replace()

### Concept Explanation
.str.replace() substitutes occurrences of a substring (or pattern) with a replacement string, across the whole Series at once. It supports both simple literal replacement and, with regex=True, full pattern-based replacement — useful for standardizing formatting, masking sensitive data, or cleaning up inconsistent text.

### Business + AI/ML Example
For privacy, support wants phone numbers **masked** in any feedback notes that get shared in a report, and the region names need a couple of legacy abbreviations expanded to their full names for a client-facing dashboard.

In [5]:
region_display = df["region"].replace({"South": "Southern Zone", "North": "Northern Zone"})
df["region_display"] = region_display
df[["region", "region_display"]].head(4)

,region,region_display
0,South,Southern Zone
1,North,Northern Zone
2,South,Southern Zone
3,West,West


**Output Explanation:** Series.replace() with a dictionary swapped whole values ("South" → "Southern Zone"), which is handy for exact-match relabeling of categorical columns like region. (Note: this is Series.replace(), not .str.replace() — the difference matters, since .str.replace() works on substrings within text, shown next.)

In [10]:
df["feedback_masked"] = df["feedback_notes"].str.replace(r"\d{5}-\d{5}", "[PHONE REDACTED]", regex=True)
df[["feedback_notes", "feedback_masked"]]

,feedback_notes,feedback_masked
0,"Great service, fast delivery! Call me at 98765...","Great service, fast delivery! Call me at [PHON..."
1,"Delivery was delayed by 3 days, not happy.","Delivery was delayed by 3 days, not happy."
2,"Product quality is excellent, will order again.","Product quality is excellent, will order again."
3,Very disappointed - delivery never arrived. Co...,Very disappointed - delivery never arrived. Co...
4,"Support team was helpful, resolved my issue qu...","Support team was helpful, resolved my issue qu..."
5,"Amazing experience, quick delivery every time!","Amazing experience, quick delivery every time!"
6,delivery delayed AGAIN. This is the second time.,delivery delayed AGAIN. This is the second time.
7,"No complaints, everything works as expected.","No complaints, everything works as expected."
8,Packaging was damaged but delivery was on time.,Packaging was damaged but delivery was on time.
9,"Terrible support, delivery delayed, want a ref...","Terrible support, delivery delayed, want a ref..."


**Output Explanation:** The pattern \d{5}-\d{5} matches phone numbers in the 98765-43210 format anywhere inside the text, and .str.replace(..., regex=True) swapped every match for "[PHONE REDACTED]" across the whole column in one call — no manual string surgery needed. Rows without a phone number are left untouched.

## 7. Regular Expressions (Regex)

### Concept Explanation
Regular expressions (regex) describe **patterns** in text rather than exact substrings — e.g., "a sequence of digits", "a word starting with a capital letter", or "an email-shaped string". Pandas string methods (.str.contains(), .str.replace(), .str.extract(), .str.findall()) all accept regex patterns, making them dramatically more powerful than plain substring matching. Common building blocks: \d (digit), \w (word character), +/* (one-or-more / zero-or-more), {n} (exactly n repeats), () (capture group).

### Business + AI/ML Example
Support wants every embedded **phone number** actually pulled out into its own column (not just masked), so agents can call customers back directly without re-reading the full feedback text. This structured phone_number column also becomes a simple "did the customer leave contact info?" feature for the churn/support pipeline.

In [6]:
df["phone_number"] = df["feedback_notes"].str.extract(r"(\d{5}-\d{5})")
df[["feedback_notes", "phone_number"]]

,feedback_notes,phone_number
0,"Great service, fast delivery! Call me at 98765...",98765-43210
1,"Delivery was delayed by 3 days, not happy.",NaN
2,"Product quality is excellent, will order again.",NaN
3,Very disappointed - delivery never arrived. Co...,91234-56789
4,"Support team was helpful, resolved my issue qu...",NaN
5,"Amazing experience, quick delivery every time!",NaN
6,delivery delayed AGAIN. This is the second time.,NaN
7,"No complaints, everything works as expected.",NaN
8,Packaging was damaged but delivery was on time.,NaN
9,"Terrible support, delivery delayed, want a ref...",90000-11111


**Output Explanation:** str.extract(r"(\d{5}-\d{5})") searched each feedback string for the pattern "5 digits, a hyphen, 5 digits" and pulled the match into a new **phone_number** column. Rows with no matching pattern correctly show **NaN** — this is a clean, structured way to lift a specific piece of information out of free-form text.

In [21]:
digit_groups = df["feedback_notes"].str.findall(r"\d+")
df["numbers_found"] = digit_groups
df[["feedback_notes", "numbers_found"]]

,feedback_notes,numbers_found
0,"Great service, fast delivery! Call me at 98765...","[98765, 43210]"
1,"Delivery was delayed by 3 days, not happy.",[3]
2,"Product quality is excellent, will order again.",[]
3,Very disappointed - delivery never arrived. Co...,"[91234, 56789]"
4,"Support team was helpful, resolved my issue qu...",[]
5,"Amazing experience, quick delivery every time!",[]
6,delivery delayed AGAIN. This is the second time.,[]
7,"No complaints, everything works as expected.",[]
8,Packaging was damaged but delivery was on time.,[]
9,"Terrible support, delivery delayed, want a ref...","[90000, 11111]"


**Output Explanation:** str.findall(r"\d+") returned a **list** of every run of digits found in each string (e.g., a phone number gets split into two number chunks by the hyphen). Rows with no digits at all show an empty list []. This is useful when a field might contain multiple pieces of numeric information that all need to be captured, not just the first.

In [22]:
urgent_pattern = r"disappoint|terrible|never arrived|refund|delayed"
df["is_negative_feedback"] = df["feedback_notes"].str.contains(urgent_pattern, case=False, regex=True, na=False)
df[["customer_name_clean", "is_negative_feedback"]]

,customer_name_clean,is_negative_feedback
0,Aarav Sharma,False
1,PRIYA verma,True
2,Rahul Nair,False
3,sneha Gupta,True
4,VIKRAM singh,False
5,Ananya Rao,False
6,karthik IYER,True
7,Divya Menon,False
8,Manoj Kumar,False
9,LAKSHMI pillai,True


**Output Explanation:** The regex pattern uses | to check for any of five negative-sentiment phrases at once, all in a single .str.contains() call. is_negative_feedback is now a ready-to-use boolean feature — this kind of lightweight, rule-based flagging is a common quick win before investing in a full NLP sentiment model.